# 02 CIFAR-10 Baseline 학습 (MSE only)

DDPM baseline. SDD 없이 MSE loss만 사용합니다.

- `num_processes=None` → GPU 자동 감지
- 단일 GPU / CPU 환경에서도 그대로 동작

In [ ]:
import torch
from src.experiments import load_cfg, deep_update

# ── GPU 자동 감지 ───────────────────────────────────────────────
n_gpu = torch.cuda.device_count()
print(f"감지된 GPU 수: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory // 1024**3} GB")
if n_gpu == 0:
    print("  (GPU 없음 — CPU로 실행됩니다)")


In [ ]:
from src.experiments import load_cfg, deep_update, launch_train

cfg = load_cfg("configs/cifar10.yaml")
cfg = deep_update(cfg, {
    "run_name": "cifar10_baseline_mse_only",
    "sdd": {
        "enabled": False,
        "lambda_distill": 0.0,
        "use_centering": False,
        "use_sharpening": False,
        "use_gating": False,
        "use_projection_head": False,
    },
    "wandb": {"enabled": True, "tags": ["cifar10", "baseline", "mse-only"]},
})

# num_processes=None → 자동으로 모든 GPU 사용
launch_train(cfg, num_processes=None, with_eval=True)


In [ ]:
# 학습 완료 후 loss curve 확인
from pathlib import Path
import pandas as pd, matplotlib.pyplot as plt

csv = Path("outputs/logs/cifar10_baseline_mse_only_history.csv")
if csv.exists():
    df = pd.read_csv(csv)
    ax = df.plot(x="epoch", y=["loss_total", "loss_mse"], figsize=(10, 4),
                 title="Baseline training curves")
    ax.grid(True)
    plt.show()
    print(df.tail())
else:
    print("아직 학습 결과 없음 — 위 셀을 먼저 실행하세요.")
